LlamaIndex's Capabilities to work with SQL databases


In [ ]:
!pip install llama-index openai


In [ ]:
#SQLAlchemy is a popular SQL toolkit and Object-Relational Mapping (ORM) library for Python. It provides a high-level interface for working with databases, allowing developers to interact with databases using Python objects and classes instead of writing raw SQL queries. SQLAlchemy supports a wide range of database backends, including SQLite, PostgreSQL, MySQL, and more.

from sqlalchemy import (
    create_engine,
    text,
)
from llama_index.core import SQLDatabase
from llama_index.llms.openai import OpenAI


In [ ]:
#set up the database connection
db_user ="root"
db_password = "####"
db_host="localhost:3306"
db_name= "demo_db"

connection_string= f"mysql+pymysql://{db_user}:{db_password}@{db_host}/{db_name}"

In [ ]:
#create an engine instance -The engine is a factory in sql alchemy that can create new connections
engine = create_engine(connection_string)

#test the connection using raw sql
print("printing three rows:")
with engine.connect() as connection:
    result = connection.execute(text("SELECT * FROM employees LIMIT 3"))
    for row in result:
        print(row)
print("printing table structure:")
with engine.connect() as connection:
    result = connection.execute(text("DESCRIBE walmart"))
    for row in result:
        print(row)

In [ ]:
llm = OpenAI(temperature = 0.1,model="gpt-3.5-turbo")

In [ ]:
#AQL database abstraction (a light wrapper around sqlAlchemy)
sql_database = SQLDatabase(engine,include_tables=["walmart"])

Part 1 : Text to sql query engine
once we construct our sql database,we can use the NLSQLTABLEQueryEngine to construct natural language queries that are synthesized into sql queries.Note that we need to specify the tables we want to use this query engine.if we don't the query engine will pull all the schema context,which could overflow the context window of the llm


In [ ]:
from llama_index.core.query_engine import NLSQLTableQueryEngine

query_engine = NLSQLTableQueryEngine(
    sql_database=sql_database,tables=["walmart"],llm=llm
)
#query_engine.query("What are the columns in the walmart table?")
query_str="what are the top 5 most expensive products in the walmart table?"
response =query_engine.query(query_str)
print(response)

Part 2 : Query-time Retrival of tables for text-to-sql
if we don't know ahead of time which table we would like to use,and the total size of the table schema overflows your context window size,we should store the table schema in an index so that during query time we can retieve the right schema.The way we can do this using the SQLTableNodeMapping object,which takes in a SQL Database and produces a Node object for each SQLTableSchema object passed into ObjectIndex constructor

In [ ]:
from llama_index.core.indices.struct_store.sql_query import(
    SQLTableRetrieverQueryEngine,
)
from llama_index.core.objects import (
    SQLTableNodeMapping,
    ObjectIndex,
    SQLTableSchema,
)
from llama_index.core import VectorStoreIndex


In [ ]:
table_node_mapping = SQLTableNodeMapping(sql_database)
table_schema_objs = [
    (SQLTableSchema(table_name="walmart"))
]#add a sql table for our table ,we can add more here 

#create an object index using the table schema objects and the table node mapping
obj_index=ObjectIndex.from_objects(
    table_schema_objs,
    table_node_mapping,
    index_cls=VectorStoreIndex,
)
query_engine=SQLTableRetrieverQueryEngine(sql_database,obj_index.as_retriver(similarity_top_k=3))


In [ ]:
query_str ="what is the average CPI of each store?order the results by store number."
response = query_engine.query(query_str)
print(response)

Part 3 : Text-to-SQL Retriver
so far text-to-SQL capability is packaged in a query engine and consists of both retrieval and synthesis.You can use the sql retriever on it's own

In [ ]:
from llama_index.core.retrievers import NLSQLRetriever


In [ ]:
#default retrival (return_raw=True)
nl_sql_retriever = NLSQLRetriever(
    sql_database,tables=["walmart"],llm=llm,return_raw=True
)

In [ ]:
#we compose our sql retriever with our standard retrieverqueryengine to synthesize a response.The result is roughly similar to our packaged text -to -SQL query engines

In [ ]:
from llama_index.core.query_engine import RetrieverQueryEngine

query_engine = RetrieverQueryEngine.from_args(nl_sql_retriever)
response = query_engine.query(
    "what is the average cpi of each store?order the results by store number"
)
print(str(response))